# Lead Reply Prediction — TheLeadFlow

Train a model to predict `replied` (0/1) from lead features.

**Pipeline:** load CSV → class balance → feature engineering → encode → train (LogReg + LightGBM) → cross-validation → hold-out evaluation → feature importance → save model.

👉 Set `CSV_PATH` below to your dataset (must contain a `replied` column).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix, RocCurveDisplay)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

import matplotlib.pyplot as plt

CSV_PATH = 'data/leads_training.csv'   # ← change to your 827-row dataset

## 1. Load data & class balance

The key feasibility factor is the **minority class** size (≥ ~50-60 ideal).

In [ ]:
df = pd.read_csv(CSV_PATH)
print('Rows:', len(df))
df.head()

In [ ]:
y = df['replied'].astype(int)
n, n1, n0 = len(y), int(y.sum()), int((y == 0).sum())
minority = min(n1, n0)
print(f'replied=1 : {n1} ({n1/n*100:.1f}%)')
print(f'replied=0 : {n0} ({n0/n*100:.1f}%)')
print(f'minority class: {minority}')
if minority < 30:
    print('\u26a0\ufe0f  < 30 minority \u2192 unreliable, collect more data.')
elif minority < 60:
    print('\u26a0\ufe0f  30-60 \u2192 baseline only, keep few features.')
else:
    print('\u2705 Enough to train a baseline.')

y.value_counts().plot(kind='bar', title='Class balance (replied)'); plt.show()

## 2. Feature engineering

Keep: `role, segment, email_source, score, email_verified`. Derive: `country, has_email, domain_type, has_funding, has_insights`. Drop identifiers/PII/leakage (`id, company, source_url, email, notes, status`).

In [ ]:
FREE_DOMAINS = {'gmail.com','yahoo.com','hotmail.com','outlook.com','icloud.com','protonmail.com'}

def build_features(df):
    df = df.copy()
    if 'email' in df:
        df['has_email'] = df['email'].notna().astype(int)
        def _dom(e):
            if not isinstance(e, str) or '@' not in e: return 'none'
            d = e.split('@')[-1].lower()
            return 'free' if d in FREE_DOMAINS else 'corporate'
        df['domain_type'] = df['email'].apply(_dom)
    else:
        df['has_email'], df['domain_type'] = 0, 'none'

    df['has_insights'] = df['insights'].notna().astype(int) if 'insights' in df else 0
    df['country'] = (df['location'].fillna('').apply(lambda s: (s.split(',')[-1].strip() or 'unknown'))
                     if 'location' in df else 'unknown')
    if 'notes' in df:
        df['has_funding'] = (df['notes'].fillna('')
            .str.contains(r'rais|fund|seed|series|\$|\u00a3|\u20ac|\bM\b', case=False, regex=True).astype(int))
    else:
        df['has_funding'] = 0
    if 'email_verified' in df:
        df['email_verified'] = df['email_verified'].astype(int)

    cat = [c for c in ['role','country','email_source','segment','domain_type'] if c in df.columns]
    num = [c for c in ['score','email_verified','has_email','has_insights','has_funding'] if c in df.columns]
    X = df[cat + num].copy()
    for c in cat: X[c] = X[c].fillna('unknown').astype(str)
    for c in num: X[c] = pd.to_numeric(X[c], errors='coerce').fillna(0)
    return X, cat, num

X, cat, num = build_features(df)
print('Categorical:', cat)
print('Numeric    :', num)
X.head()

## 3. Train & cross-validate

Stratified 5-fold CV. `class_weight='balanced'` handles the imbalance.

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=False)
pre = ColumnTransformer([('cat', ohe, cat), ('num', StandardScaler(), num)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000)}
if HAS_LGBM:
    models['LightGBM'] = LGBMClassifier(n_estimators=300, num_leaves=15, min_child_samples=20,
                                        learning_rate=0.05, class_weight='balanced', verbose=-1)

best_name, best_auc, best_pipe = None, -1, None
for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    auc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc').mean()
    ap  = cross_val_score(pipe, X, y, cv=cv, scoring='average_precision').mean()
    print(f'{name:20} ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}')
    if auc > best_auc:
        best_name, best_auc, best_pipe = name, auc, pipe
print('\nBest:', best_name, f'(ROC-AUC={best_auc:.3f})')

## 4. Hold-out evaluation

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
best_pipe.fit(Xtr, ytr)
proba = best_pipe.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
print('ROC-AUC:', round(roc_auc_score(yte, proba), 3), ' PR-AUC:', round(average_precision_score(yte, proba), 3))
print(confusion_matrix(yte, pred))
print(classification_report(yte, pred, digits=3))
RocCurveDisplay.from_predictions(yte, proba); plt.show()

## 5. Feature importance

In [ ]:
best_pipe.fit(X, y)
feat_names = best_pipe.named_steps['pre'].get_feature_names_out()
clf = best_pipe.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
else:
    imp = np.abs(clf.coef_[0])
fi = pd.Series(imp, index=feat_names).sort_values(ascending=False).head(15)
fi.plot(kind='barh', title='Top features').invert_yaxis(); plt.show()
fi

## 6. Save model & score leads

`proba * 100` = lead score 0-100 (probability of reply).

In [ ]:
joblib.dump({'pipeline': best_pipe, 'features': cat + num}, 'ml/reply_model.joblib')
print('Saved \u2192 ml/reply_model.joblib')

# Example: score the dataset
scores = (best_pipe.predict_proba(X)[:, 1] * 100).round(1)
df_out = df.assign(reply_score=scores).sort_values('reply_score', ascending=False)
df_out[['company', 'role', 'segment', 'reply_score', 'replied']].head(10)